In [ ]:
import random
import os
from datasets import load_dataset, get_dataset_config_names
# 필요한 라이브러리만 불러옵니다. (이 코드는 데이터셋의 구조를 이해하는 데 중점을 둡니다.)

# ===========================================================================
# 🏆 Tutorial Goal: ExamBench 데이터셋 탐험하기 (AI 코딩 첫걸음)
# 💡 데이터셋명: shravan01/exambench
# ✨ 의미: 경쟁 시험 대비를 위한 대규모 AI 코퍼스입니다. (JEE, GRE, UPSC 등)
# 📋 구조 분석: 이 데이터셋은 단순 질문-답변을 넘어, '사고 과정(Chain-of-Thought)'
#             까지 학습하게 만든 고급 자료입니다. 초보자도 AI의 사고 흐름을 따라잡는
#             가장 재미있는 연습이 될 거예요!
# ===========================================================================

# 1. 환경 설정 및 데이터 로드 전략 (매우 중요!)
DATASET_NAME = "shravan01/exambench"
SAMPLE_COUNT = 5  # 처음 5개의 샘플만 가지고 실습을 진행합니다. 너무 크면 로딩 시간이 길어요!

print("==========================================================")
print("✨ [Step 1] 데이터셋 로드 준비: ExamBench와 인사해요!")
print("==========================================================")

dataset = None
sample_data_list = []

# A. 스트리밍 모드로 로드 시도 (가장 효율적이고 빠른 방법!)
try:
    print("🚀 시도: 스트리밍 모드 (Streaming=True)로 데이터셋을 로드합니다...")
    # 스트리밍 모드는 데이터를 chunk 단위로 처리하므로 메모리를 아끼고 빠릅니다.
    dataset = load_dataset(DATASET_NAME, split='train', streaming=True)
    print("✅ 성공: 스트리밍 방식으로 데이터 로드가 준비되었습니다!")
except Exception as e:
    print(f"⚠️ 오류 발생: 스트리밍 로드 실패 ({e.__class__.__name__}). 일반 모드로 전환합니다.")
    # B. 스트리밍이 안 될 경우, 일반 Dataset 객체로 다운로드 (학습용 작은 샘플만 받기)
    try:
        print("💾 일반 모드 로드: 작은 샘플만 다운로드하여 진행합니다...")
        dataset = load_dataset(DATASET_NAME, split='train')
        print("✅ 성공: 일반 Dataset 객체로 데이터 로드가 완료되었습니다.")
    except Exception as e_fallback:
        print(f"❌ 치명적 오류: 데이터셋을 로드할 수 없습니다. 오류: {e_fallback}")
        exit()

# C. 데이터 샘플 추출 (Constraint 2, 7, 16 준수)
# 스트리밍 모드와 일반 모드 모두에서 안전하게 샘플을 가져오는 패턴을 사용합니다.
print(f"\n⏳ {SAMPLE_COUNT}개의 샘플을 가져와 실습을 준비합니다...")

if hasattr(dataset, "take"):
    # .take()가 존재하면 스트리밍 데이터셋 (IterableDataset)로 간주
    sample_iterator = dataset.take(SAMPLE_COUNT)
    sample_data_list = list(sample_iterator)
else:
    # 일반 데이터셋 (Dataset)일 경우
    sample_data_list = list(dataset.select(range(min(SAMPLE_COUNT, len(dataset)))))

if sample_data_list:
    print(f"✨ 준비 완료! 총 {len(sample_data_list)}개의 샘플을 사용합니다.")
else:
    print("😭 샘플 데이터를 가져오는 데 실패하여 실습을 진행할 수 없습니다.")
    exit()


# ===========================================================================
# 🌟 실습 1: 'AI 사고 과정' 분석기 구현 (The Chain-of-Thought Analyzer)
# ===========================================================================
print("\n" + "="*70)
print("🤖 [Step 2] AI 사고 과정(CoT) 추론 분석기 시뮬레이션")
print("🎯 목표: AI가 어떻게 '생각'하고 '결론'에 도달하는지 추적합니다.")
print("==========================================================")

print("\n🧠 **튜터의 설명**: 이 데이터셋의 핵심은 `prompt` (질문)와 `complex_cot` (사고 과정)입니다.")
print("    좋은 AI는 이 과정을 논리적으로 따라가야 하죠! 직접 분석해 봅시다.")

for i, sample in enumerate(sample_data_list):
    print("\n" + "★" * 20 + f"\n\n[ ✨ 문제 #{i+1} 분석 시작 ✨ ]" + "\n" + "★" * 20)

    # 1. 기본 정보 출력 및 개요 파악 (Question Analysis)
    print(f"\n💡 1. 주어진 문제 (Prompt):\n{sample['prompt']}"[:100] + "...")

    # 2. 핵심 로직: CoT 분석 (The Reasoning)
    cot = sample['complex_cot']
    print("\n✅ 2. AI가 제시한 사고 과정 (Chain-of-Thought, CoT):")
    
    # CoT가 너무 길면 일부만 보여주어 가독성을 높입니다.
    if len(cot) > 500:
        print(cot[:400] + "...")
    else:
        print(cot)
    
    # 3. 최종 결과 도출 (The Answer)
    response = sample['response']
    print("\n💡 3. 최종 응답/정답 (Response):")
    print(f"   -> [{response}]")

    # 4. 간단한 메타 분석 (Quantitative Insight)
    print("\n--- [ 🔎 데이터 구조 분석 ] ---")
    print(f"  * Prompt 길이 추정: {len(sample['prompt'])} 문자")
    print(f"  * CoT 길이 추정: {len(cot)} 문자 (가장 길죠? AI의 깊이를 보여줍니다!)")
    print(f"  * 응답 길이 추정: {len(response)} 문자")
    print("-" * 30)


# ===========================================================================
# 🌟 실습 2: 만약 내가 문제 생성을 한다면? (Prompt Engineering Simulation)
# ===========================================================================
print("\n\n========================================================")
print("🛠️ [Step 3] 나만의 문제 생성기 시뮬레이션 (Prompt Engineering)")
print("🎯 목표: 구조화된 데이터에서 핵심 질문만 뽑아내고 패턴을 찾아봅니다.")
print("========================================================")

# 이 데이터셋은 already 구조화되어 있지만, 우리가 '사용'하는 방법을 연습합니다.
print("\n🤔 **튜터의 팁**: 실제 AI 개발에서는 '어떤 질문'을 던지느냐가 핵심입니다.")
print("    우리는 CoT가 풍부한 샘플들을 활용해, '깊이 있는 질문' 패턴을 익혀야 해요.")

# 첫 번째 샘플을 대상으로, 만약 이 데이터셋을 이용해 LLM 프롬프트를 만든다면?
if sample_data_list:
    sample = sample_data_list[0]
    
    # 새로운 프롬프트 구성 (Context + Question)
    print("\n✨ [시뮬레이션] 우리가 만들어 볼 새로운 프롬프트입니다.")
    
    # 1. 배경지식 제공 (Context): '이것은 대학수학 문제 풀이 문제입니다.'
    context_template = "당신은 명문대의 수학 문제입니다. 다음 조건을 바탕으로 단계별 풀이를 제시하세요.\n"
    
    # 2. 핵심 질문 삽입
    user_question = sample['prompt']
    
    # 3. 요청 형식 정의 (Format Instruction)
    format_instruction = "\n[요구 사항]: 답변은 반드시 단계별 사고 과정(Chain-of-Thought)을 포함해야 합니다."
    
    final_prompt = context_template + user_question + format_instruction
    
    print("\n================================================")
    print("📝 최종 LLM 프롬프트 예시 (출력 형식):")
    print("================================================")
    print(final_prompt)
    print("\n[💡 다음 단계]: 이 프롬프트를 ChatGPT 같은 LLM에게 넣고, 정답과 비교하는 연습을 하면 최고의 학습이 됩니다!")
else:
    print("실습 2 진행 불가: 샘플 데이터가 준비되지 않았습니다.")

print("\n" + "="*70)
print("🎉 끝! 🎉 초급 AI 코딩 실습을 성공적으로 마쳤습니다!")
print("데이터셋의 구조와 AI의 사고 흐름을 이해하는 것이 가장 중요해요. 수고하셨습니다!")
print("="*70)